# 02a m9_pbm Method And Worked Example

**Research question.** How does `m9_pbm` turn an N-shaped reverse-power-flow
sign error into a physically interpretable candidate correction?

This notebook explains the method before any model comparison is performed. It
uses one fixed, previously chosen example: **Alpha substation F on 17 February
2024**. The notebook reads the final Alpha Parquet file, checks its hash and
schema, reconstructs the two possible underlying demand curves from the
equations below, and writes one two-panel publication figure.

**Inputs:** final Alpha data and the versioned experiment configuration.  
**Outputs:** one plot-data CSV, one 300-dpi PNG, and one reproducibility manifest.  
**Expected runtime:** under one minute on a laptop.

## 1. Imports, Paths, And Visible Configuration

The path search below starts from the current working directory, so this
notebook can run on another laptop after cloning the repository. The displayed
configuration is the source of truth for the candidate bounds and feature
constants used throughout Notebooks 02a-02g.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    load_dataset,
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    validate_input_hashes,
    write_csv,
    write_manifest,
)
from _m9_pbm_features import (  # noqa: E402
    CandidateSpec,
    bridge_line,
    reconstruct_demand,
)
from _m9_pbm_plotting import plot_method_example  # noqa: E402

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SPEC = CandidateSpec.from_config(CONFIG)
SLUG = "02a_m9_pbm_method_example"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)

display(pd.Series(CONFIG["m9_pbm"]["example"], name="worked_example"))
display(pd.Series(CONFIG["m9_pbm"]["candidate_windows"], name="candidate_windows"))

substation_id       alpha_F
date             2024-02-17
Name: worked_example, dtype: object

slots_per_day              96
slot_minutes               15
scan_start_slot            24
scan_end_slot              72
min_duration_slots          2
max_duration_slots         32
shoulder_slots              3
anchor_offset_slots         1
solar_peak_radius_slots    14
max_internal_gap_slots      4
Name: candidate_windows, dtype: int64

## 2. Input Validation

The final dataset hash must match the configuration before results are written.
For the worked example we also require exactly 96 quarter-hour readings, one
positive day label, and 16 positive interval labels. These checks prevent a
silent label refresh or an incomplete day from changing the figure.

In [2]:
hash_audit = validate_input_hashes(PATHS, CONFIG)
display(hash_audit)

alpha = load_dataset("alpha", article_root=ARTICLE_ROOT, config=CONFIG)
example_cfg = CONFIG["m9_pbm"]["example"]
example = alpha.loc[
    alpha["substation_id"].eq(example_cfg["substation_id"])
    & alpha["date"].eq(example_cfg["date"])
].copy()
example = example.sort_values("timestamp").reset_index(drop=True)
example["slot"] = example["timestamp"].dt.hour * 4 + example["timestamp"].dt.minute // 15

assert len(example) == 96, f"Expected 96 readings, found {len(example)}."
assert example["slot"].nunique() == 96, "The example does not contain 96 unique slots."
assert bool(example["label_day"].max())
assert int(example["label_interval"].sum()) == 16

input_audit = pd.DataFrame(
    [{
        "substation_id": example_cfg["substation_id"],
        "date": example_cfg["date"],
        "readings": len(example),
        "labelled_rpf_intervals": int(example["label_interval"].sum()),
        "first_labelled_time": example.loc[example["label_interval"], "timestamp"].min(),
        "last_labelled_time": example.loc[example["label_interval"], "timestamp"].max(),
    }]
)
display(input_audit)

,filename,expected_sha256,actual_sha256,matches
0,dataset_alpha.parquet,c2d0982d138a1faebd30df60b7fd861c73007b5709187a...,c2d0982d138a1faebd30df60b7fd861c73007b5709187a...,True
1,dataset_beta.parquet,e1971eff266de145c4aca79ce5272541afee5861d18874...,e1971eff266de145c4aca79ce5272541afee5861d18874...,True
2,dataset_gamma.parquet,7ebeedeab63b1a95059c0f413744230a183bf1d6f6883f...,7ebeedeab63b1a95059c0f413744230a183bf1d6f6883f...,True


,substation_id,date,readings,labelled_rpf_intervals,first_labelled_time,last_labelled_time
0,alpha_F,2024-02-17,96,16,2024-02-17 10:00:00+00:00,2024-02-17 13:45:00+00:00


## 3. Physical Interpretation And Core Equations

During a true reverse-power-flow interval, net load should be negative because
solar export exceeds local demand. A polarity or sign error instead makes this
export appear as a positive midday bump, often with an N-shaped rise and fall.
`m9_pbm` asks which of two interpretations produces the more physically
plausible full-day underlying demand curve.

The **no-correction interpretation** is

$$
U_{no}(t) = S(t) + y(t).
$$

For a candidate window $W$, the **candidate-corrected interpretation** is

$$
U_{corr,W}(t) =
\begin{cases}
S(t)-y(t), & t \in W,\\
S(t)+y(t), & t \notin W.
\end{cases}
$$

If the day is classified as positive, its corrected net-load output is

$$
y_{corr,W}(t) =
\begin{cases}
-y(t), & t \in W,\\
y(t), & t \notin W.
\end{cases}
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $t$ | One 15-minute timestamp in a substation-day. |
| $y(t)$ | Observed net load in MW. A sign error appears as an incorrectly positive RPF bump. |
| $S(t)$ | Estimated solar generation in MW. |
| $W$ | One contiguous candidate correction window. |
| $U_{no}(t)$ | Reconstructed underlying demand if no sign correction is applied. |
| $U_{corr,W}(t)$ | Reconstructed demand if the sign is corrected only inside $W$. |
| $y_{corr,W}(t)$ | Net load written after a positive day decision. |

## 4. Candidate Windows, Features, And Decisions

Candidate generation is deterministic and uses no labels. It scans the daytime
range from 06:00 through 18:00. A window must last from 2 to 32 slots (30
minutes to 8 hours), and its midpoint must be within 14 slots (3.5 hours) of
that day's solar peak. Each candidate is evaluated on the full-day
reconstructions, with local calculations focused on $W$ and its shoulders
$\Omega(W)$.

For active feature set $A$, candidate selection and day classification are
separate operations:

$$
Score(W)=\sum_{i\in A}w_iF_i(W),
\qquad
W_d^*=\operatorname*{arg\,max}_{W} Score(W),
$$

$$
\widehat{RPF}(d)=\mathbb{1}\{Score(W_d^*)\geq\tau\}.
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $d$ | One substation-day. |
| $A$ | The active set of physical features. |
| $F_i(W)$ | Value of feature $i$ for candidate $W$. |
| $w_i$ | Nonnegative weight assigned to feature $i$. |
| $Score(W)$ | Weighted physical plausibility score for candidate $W$. |
| $W_d^*$ | Highest-scoring candidate on day $d$; ties are resolved deterministically. |
| $\tau$ | Day threshold learned from training substations only. |
| $\mathbb{1}\{\cdot\}$ | Indicator equal to one when its condition is true. |

Only when $\widehat{RPF}(d)=1$ does $W_d^*$ become the predicted correction
interval. A negative day decision leaves every observed reading unchanged.

## 5. The Nine Physical Feature Concepts

The features describe complementary physical evidence. Positive improvement
features mean that the candidate correction makes demand more plausible.

### F1: bridge improvement

Let $L_W(t)$ be the straight line joining demand immediately before and after
$W$:

$$
E_{bridge}(U,W)=\operatorname{median}_{t\in W}|U(t)-L_W(t)|,
$$

$$
F_1(W)=\frac{E_{bridge}(U_{no},W)-E_{bridge}(U_{corr,W},W)}
{E_{bridge}(U_{no},W)+E_{bridge}(U_{corr,W},W)+\epsilon}.
$$

Here $L_W(t)$ is the linear bridge, $E_{bridge}$ is median absolute bridge
error, and $\epsilon$ is a small positive constant that prevents division by
zero.

### F2: roughness improvement

$$
TV(U,\Omega(W))=\sum_{t,t+\Delta t\in\Omega(W)}|U(t+\Delta t)-U(t)|,
$$

$$
F_2(W)=\frac{TV(U_{no},\Omega(W))-TV(U_{corr,W},\Omega(W))}
{TV(U_{no},\Omega(W))+TV(U_{corr,W},\Omega(W))+\epsilon}.
$$

$\Omega(W)$ is $W$ plus three-slot shoulders, $\Delta t$ is 15 minutes, and
$TV$ is total variation over adjacent readings.

### F3: slope-continuity improvement

$$
J(U,W)=|m_{out,left}(U)-m_{in,left}(U)|
+|m_{in,right}(U)-m_{out,right}(U)|,
$$

$$
F_3(W)=\frac{J(U_{no},W)-J(U_{corr,W},W)}
{J(U_{no},W)+J(U_{corr,W},W)+\epsilon}.
$$

$m_{out,left}$ and $m_{out,right}$ are robust slopes outside the boundaries;
$m_{in,left}$ and $m_{in,right}$ are slopes just inside them; $J$ is their
summed mismatch.

### F4-F7: duration, N-height, solar strength, and peak alignment

$$
F_4(W)=\operatorname{clip}\left(\frac{duration_{hours}(W)}{1.5},0,1\right),
$$

$$
F_5(W)=\operatorname{clip}\left(
\frac{\max_{t\in W}y(t)-\max(y(t_{left}),y(t_{right}))}{P_{scale}},0,1\right),
$$

$$
F_6(W)=\operatorname{clip}\left(\frac{P95_{t\in W}S(t)}{S_{substation}},0,1\right),
$$

$$
F_7(W)=\operatorname{clip}\left(
1-\frac{|midpoint(W)-t_{solar\ peak}|}{3.5\ hours},0,1\right).
$$

$duration_{hours}$ is window length; $t_{left}$ and $t_{right}$ are its boundary
slots; $P_{scale}$ is a robust day power scale; $P95$ is the 95th percentile;
$S_{substation}$ is the substation's median historical daytime-solar P95; and
$t_{solar\ peak}$ is that day's maximum-solar slot.

### F8-F9: substation-relative core score

$$
core(W)=F_1(W)+F_2(W)+F_3(W),
$$

$$
F_8(W)=robust\_bound\{core(W)-median(core_{best,day})\},
$$

$$
F_9(d)=2\,percentile\_rank(core_{best,d})-1.
$$

$core_{best,d}$ is the highest core score on day $d$. The median and percentile
rank are calculated within the same substation using unlabelled scores only.
F8 measures centred magnitude; F9 measures relative daily rank. Neither uses a
manual RPF label.

## 6. Reconstruct The Worked Example

The labelled Alpha interval is used here only to define the worked window. It
runs from slot 40 through slot 55, or 10:00 through 13:45 inclusive. The bridge
anchors are the immediately adjacent readings at 09:45 and 14:00. Later model
notebooks generate and select candidates without reading these labels.

In [3]:
net_load = example["net_load_MW"].to_numpy(dtype=float)
solar = example["solar_MW"].to_numpy(dtype=float)
true_slots = example.loc[example["label_interval"], "slot"].to_numpy(dtype=int)
left_slot, right_slot = int(true_slots.min()), int(true_slots.max())
assert (left_slot, right_slot) == (40, 55)

uncorrected, corrected = reconstruct_demand(net_load, solar, left_slot, right_slot)
bridge_inside, (left_anchor, right_anchor) = bridge_line(
    uncorrected, left_slot, right_slot, SPEC
)
bridge_slots = np.arange(left_anchor, right_anchor + 1)
bridge_values = np.interp(
    bridge_slots,
    [left_anchor, right_anchor],
    [uncorrected[left_anchor], uncorrected[right_anchor]],
)

plot_data = pd.DataFrame(
    {
        "substation_id": example["substation_id"],
        "date": example["date"],
        "timestamp": example["timestamp"],
        "slot": example["slot"],
        "observed_net_load_MW": net_load,
        "solar_generation_MW": solar,
        "uncorrected_demand_MW": uncorrected,
        "corrected_demand_MW": corrected,
        "true_interval": example["label_interval"],
        "candidate_window": example["slot"].between(left_slot, right_slot),
        "bridge_anchor": example["slot"].isin([left_anchor, right_anchor]),
        "linear_bridge_MW": np.nan,
    }
)
plot_data.loc[plot_data["slot"].isin(bridge_slots), "linear_bridge_MW"] = bridge_values

assert np.allclose(
    plot_data.loc[plot_data["candidate_window"], "linear_bridge_MW"],
    bridge_inside,
)
display(
    pd.Series(
        {
            "candidate_start": plot_data.loc[plot_data["candidate_window"], "timestamp"].min(),
            "candidate_end": plot_data.loc[plot_data["candidate_window"], "timestamp"].max(),
            "left_anchor": plot_data.loc[plot_data["slot"].eq(left_anchor), "timestamp"].iloc[0],
            "right_anchor": plot_data.loc[plot_data["slot"].eq(right_anchor), "timestamp"].iloc[0],
        },
        name="worked_window",
    )
)

candidate_start   2024-02-17 10:00:00+00:00
candidate_end     2024-02-17 13:45:00+00:00
left_anchor       2024-02-17 09:45:00+00:00
right_anchor      2024-02-17 14:00:00+00:00
Name: worked_window, dtype: datetime64[ns, UTC]

## 7. Publication Figure And Source Data

Panel (a) retains the measurements the method actually sees. Panel (b) changes
only the demand interpretation inside the candidate window. The bridge is
anchored outside that window, so it provides a local reference without
flattening or replacing the reconstructed demand curve.

In [4]:
TABLE_PATH = OUTPUT_DIRS["tables"] / "table01_alpha_F_2024-02-17_plot_data.csv"
FIGURE_PATH = OUTPUT_DIRS["figures"] / "fig01_m9_pbm_alpha_F_2024-02-17.png"

write_csv(plot_data, TABLE_PATH)
plot_method_example(plot_data, FIGURE_PATH)

display(plot_data.head())
display(FIGURE_PATH)

,substation_id,date,timestamp,slot,observed_net_load_MW,solar_generation_MW,uncorrected_demand_MW,corrected_demand_MW,true_interval,candidate_window,bridge_anchor,linear_bridge_MW
0,alpha_F,2024-02-17,2024-02-17 00:00:00+00:00,0,5.589320,0.005066,5.594386,5.594386,False,False,False,NaN
1,alpha_F,2024-02-17,2024-02-17 00:15:00+00:00,1,6.359163,0.004944,6.364107,6.364107,False,False,False,NaN
2,alpha_F,2024-02-17,2024-02-17 00:30:00+00:00,2,6.191647,0.003627,6.195274,6.195274,False,False,False,NaN
3,alpha_F,2024-02-17,2024-02-17 00:45:00+00:00,3,5.888226,0.003387,5.891613,5.891613,False,False,False,NaN
4,alpha_F,2024-02-17,2024-02-17 01:00:00+00:00,4,5.794897,0.004131,5.799028,5.799028,False,False,False,NaN


WindowsPath('C:/Users/z5404477/Documents/PyNRPF/publication/2_journal_article/outputs/figures/02a_m9_pbm_method_example/fig01_m9_pbm_alpha_F_2024-02-17.png')

## 8. Findings And Limitations

For this worked day, the positive midday net-load bump creates an implausible
feature in the no-correction demand curve. Applying the candidate sign change
from 10:00 through 13:45 yields a smoother physical interpretation relative to
the two outside anchors. This figure demonstrates the mechanism; it is not a
performance estimate.

The example window comes from the Alpha reference label and must not be confused
with a fitted model prediction. Candidate selection, substation-held-out
threshold selection, and interval evaluation are performed in later notebooks.

## 9. Reproducibility Manifest And Output Inventory

The manifest records current input hashes, configuration hash, environment,
elapsed time, row counts, and output hashes. Only declared final inputs are read.

In [5]:
manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=[PATHS.config, PATHS.final_data / "dataset_alpha.parquet"],
    outputs=[TABLE_PATH, FIGURE_PATH],
    row_counts={
        "example_readings": len(plot_data),
        "labelled_intervals": int(plot_data["true_interval"].sum()),
    },
)
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)

output_inventory = pd.DataFrame(
    {
        "type": ["table", "figure", "manifest"],
        "path": [TABLE_PATH, FIGURE_PATH, MANIFEST_PATH],
    }
)
output_inventory["exists"] = output_inventory["path"].map(Path.exists)
output_inventory["bytes"] = output_inventory["path"].map(lambda path: path.stat().st_size)
display(output_inventory)
assert output_inventory["exists"].all()
assert (output_inventory["bytes"] > 0).all()

,type,path,exists,bytes
0,table,C:\Users\z5404477\Documents\PyNRPF\publication...,True,13109
1,figure,C:\Users\z5404477\Documents\PyNRPF\publication...,True,333160
2,manifest,C:\Users\z5404477\Documents\PyNRPF\publication...,True,1405
